In [2]:
import os
base_dir = "/kaggle/working/"
os.chdir(base_dir)
os.makedirs("TextSummarization/data/raw", exist_ok=True)
os.makedirs("TextSummarization/data/processed", exist_ok=True)
os.makedirs("TextSummarization/preprocessing", exist_ok=True)
os.makedirs("TextSummarization/results", exist_ok=True)
import shutil

# Define source and destination paths
input_dir = "/kaggle/input/text-summarization1/"
raw_dir = "TextSummarization/data/raw/"

# List of files to transfer
files = ["train.csv", "test.csv", "validation.csv"]

# Copy each file
for file in files:
    src = os.path.join(input_dir, file)
    dst = os.path.join(raw_dir, file)
    shutil.copy(src, dst)

print("Files successfully copied to raw directory.")

Files successfully copied to raw directory.


In [4]:
!pip install contractions --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 6.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.3/118.3 kB 10.1 MB/s eta 0:00:00


In [5]:
import re
import pandas as pd
from tqdm import tqdm
from contractions import fix

def clean_text(text: str) -> str:
    """
    Cleans a single string by:
    1. Lowercasing the text.
    2. Expanding contractions (e.g., "don't" -> "do not").
    3. Removing HTML tags.
    4. Removing special characters and extra spaces.

    Args:
        text (str): The input text to clean.

    Returns:
        str: The cleaned text.
    """
    if not isinstance(text, str):
        return "" # Return empty string for non-string inputs, or handle as needed
    text = text.lower()
    text = fix(text)  # Expand contractions
    text = re.sub(r'<[^>]+>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

def normalize_data(input_path: str, output_path: str, sample_size: int = None):
    """
    Normalizes the 'article' and 'highlights' columns in a CSV file.

    Args:
        input_path (str): Path to the raw CSV file.
        output_path (str): Path to save the normalized CSV file.
        sample_size (int, optional): Number of rows to sample. If None, process all rows.
    """
    print(f"Normalizing data from {input_path}...")
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}")
        return
    except Exception as e:
        print(f"Error reading CSV {input_path}: {e}")
        return

    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"  Sampling {sample_size} rows.")

    # Apply cleaning to 'article' and 'highlights' columns
    tqdm.pandas(desc="Cleaning articles")
    df['article'] = df['article'].progress_apply(clean_text)

    tqdm.pandas(desc="Cleaning highlights")
    df['highlights'] = df['highlights'].progress_apply(clean_text)

    try:
        df.to_csv(output_path, index=False)
        print(f"Normalized data saved to {output_path}")
    except Exception as e:
        print(f"Error saving normalized data to {output_path}: {e}")

if __name__ == "__main__":
    # Define paths (relative to /kaggle/working/TextSummarization/)
    BASE_DIR = "/kaggle/working/TextSummarization"
    RAW_DATA_DIR = f"{BASE_DIR}/data/raw"
    PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"

    # Ensure directories exist
    import os
    os.makedirs(RAW_DATA_DIR, exist_ok=True)
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    # Normalize train, test, and validation datasets
    # For training and validation, use a sample size as specified in the prompt
    normalize_data(f"{RAW_DATA_DIR}/train.csv", f"{PROCESSED_DATA_DIR}/train_normalized.csv", sample_size=20000)
    normalize_data(f"{RAW_DATA_DIR}/validation.csv", f"{PROCESSED_DATA_DIR}/validation_normalized.csv", sample_size=20000)
    # For test, use the full dataset
    normalize_data(f"{RAW_DATA_DIR}/test.csv", f"{PROCESSED_DATA_DIR}/test_normalized.csv", sample_size=None)


Normalizing data from /kaggle/working/TextSummarization/data/raw/train.csv...
  Sampling 20000 rows.


Cleaning highlights: 100%|██████████| 20000/20000 [00:00<00:00, 25595.20it/s]


Normalized data saved to /kaggle/working/TextSummarization/data/processed/train_normalized.csv
Normalizing data from /kaggle/working/TextSummarization/data/raw/validation.csv...


Cleaning highlights: 100%|██████████| 13368/13368 [00:00<00:00, 22770.68it/s]


Normalized data saved to /kaggle/working/TextSummarization/data/processed/validation_normalized.csv
Normalizing data from /kaggle/working/TextSummarization/data/raw/test.csv...


Cleaning highlights: 100%|██████████| 11490/11490 [00:00<00:00, 23827.35it/s]


Normalized data saved to /kaggle/working/TextSummarization/data/processed/test_normalized.csv


In [6]:
import os
import pandas as pd
import sentencepiece as spm
import torch
from tqdm import tqdm

# Define constants
MAX_LEN = 64
VOCAB_SIZE = 30000

# Define paths
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model"
TOKENIZER_VOCAB_PATH = f"{BASE_DIR}/tokenizer.vocab"

def train_sentencepiece_tokenizer(data_paths: list[str], model_prefix: str, vocab_size: int):
    """
    Trains a SentencePiece tokenizer on the combined text from specified data paths.

    Args:
        data_paths (list[str]): List of paths to CSV files containing 'article' and 'highlights'.
        model_prefix (str): Prefix for the SentencePiece model files.
        vocab_size (int): Desired vocabulary size.
    """
    print(f"Training SentencePiece tokenizer with vocab size {vocab_size}...")
    temp_text_file = f"{model_prefix}_temp_corpus.txt"

    # Combine all text from articles and highlights into a single file
    all_text = []
    for path in data_paths:
        try:
            df = pd.read_csv(path)
            all_text.extend(df['article'].dropna().tolist())
            all_text.extend(df['highlights'].dropna().tolist())
        except FileNotFoundError:
            print(f"Warning: File not found at {path}. Skipping.")
        except Exception as e:
            print(f"Error reading CSV {path}: {e}")
            continue

    if not all_text:
        print("Error: No text data found to train tokenizer.")
        return

    with open(temp_text_file, 'w', encoding='utf-8') as f:
        for text in tqdm(all_text, desc="Writing corpus for tokenizer training"):
            f.write(text + '\n')

    # Train SentencePiece model
    spm.SentencePieceTrainer.train(
        f'--input={temp_text_file} --model_prefix={model_prefix} '
        f'--vocab_size={vocab_size} --character_coverage=1.0 '
        f'--model_type=unigram --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3'
    )
    print(f"SentencePiece tokenizer trained and saved as {model_prefix}.model")
    os.remove(temp_text_file) # Clean up temporary file

def load_tokenizer(model_path: str):
    """
    Loads a pre-trained SentencePiece tokenizer.

    Args:
        model_path (str): Path to the SentencePiece model file.

    Returns:
        spm.SentencePieceProcessor: Loaded tokenizer.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Tokenizer model not found at {model_path}")
    sp = spm.SentencePieceProcessor()
    sp.load(model_path)
    return sp

def tokenize_data(sp_tokenizer, input_path: str, output_path: str, max_len: int):
    """
    Tokenizes 'article' and 'highlights' columns using the provided tokenizer,
    pads/truncates sequences, and saves the data as a PyTorch tensor.

    Args:
        sp_tokenizer: Trained SentencePiece tokenizer.
        input_path (str): Path to the normalized CSV file.
        output_path (str): Path to save the tokenized data as a PyTorch .pt file.
        max_len (int): Maximum sequence length for padding/truncation.
    """
    print(f"Tokenizing data from {input_path}...")
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}. Skipping tokenization.")
        return
    except Exception as e:
        print(f"Error reading CSV {input_path}: {e}")
        return

    articles_tokenized = []
    highlights_tokenized = []

    # Tokenize and pad/truncate
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing and padding"):
        article_ids = sp_tokenizer.encode_as_ids(str(row['article']))
        highlights_ids = sp_tokenizer.encode_as_ids(str(row['highlights']))

        # Add EOS token and pad/truncate
        article_ids = article_ids[:max_len-1] + [sp_tokenizer.eos_id()]
        highlights_ids = highlights_ids[:max_len-1] + [sp_tokenizer.eos_id()]

        article_ids = article_ids + [sp_tokenizer.pad_id()] * (max_len - len(article_ids))
        highlights_ids = highlights_ids + [sp_tokenizer.pad_id()] * (max_len - len(highlights_ids))

        articles_tokenized.append(article_ids[:max_len])
        highlights_tokenized.append(highlights_ids[:max_len])

    # Convert to PyTorch tensors
    articles_tensor = torch.tensor(articles_tokenized, dtype=torch.long)
    highlights_tensor = torch.tensor(highlights_tokenized, dtype=torch.long)

    # Save as .pt file
    processed_data = {
        'articles': articles_tensor,
        'highlights': highlights_tensor,
        'pad_id': sp_tokenizer.pad_id(),
        'bos_id': sp_tokenizer.bos_id(),
        'eos_id': sp_tokenizer.eos_id(),
        'vocab_size': sp_tokenizer.get_piece_size()
    }
    try:
        torch.save(processed_data, output_path)
        print(f"Tokenized data saved to {output_path}")
    except Exception as e:
        print(f"Error saving tokenized data to {output_path}: {e}")


if __name__ == "__main__":
    # Ensure directories exist (redundant but good for standalone execution)
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    # Paths to normalized data
    train_normalized_path = f"{PROCESSED_DATA_DIR}/train_normalized.csv"
    validation_normalized_path = f"{PROCESSED_DATA_DIR}/validation_normalized.csv"
    test_normalized_path = f"{PROCESSED_DATA_DIR}/test_normalized.csv"

    # Train tokenizer using combined train and validation normalized data
    # (Using normalized data as corpus for tokenizer training)
    train_sentencepiece_tokenizer(
        data_paths=[train_normalized_path, validation_normalized_path],
        model_prefix=f"{BASE_DIR}/tokenizer",
        vocab_size=VOCAB_SIZE
    )

    # Load the trained tokenizer
    sp_tokenizer_model = load_tokenizer(TOKENIZER_MODEL_PATH)

    # Tokenize and save each dataset
    tokenize_data(sp_tokenizer_model, train_normalized_path, f"{PROCESSED_DATA_DIR}/train_tokenized.pt", MAX_LEN)
    tokenize_data(sp_tokenizer_model, validation_normalized_path, f"{PROCESSED_DATA_DIR}/validation_tokenized.pt", MAX_LEN)
    tokenize_data(sp_tokenizer_model, test_normalized_path, f"{PROCESSED_DATA_DIR}/test_tokenized.pt", MAX_LEN)

    # Example of how to get special token IDs for later use
    print(f"PAD ID: {sp_tokenizer_model.pad_id()}")
    print(f"BOS ID: {sp_tokenizer_model.bos_id()}")
    print(f"EOS ID: {sp_tokenizer_model.eos_id()}")
    print(f"VOCAB SIZE: {sp_tokenizer_model.get_piece_size()}")


Training SentencePiece tokenizer with vocab size 30000...


Writing corpus for tokenizer training: 100%|██████████| 66736/66736 [00:00<00:00, 382244.39it/s]
sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=/kaggle/working/TextSummarization/tokenizer_temp_corpus.txt --model_prefix=/kaggle/working/TextSummarization/tokenizer --vocab_size=30000 --character_coverage=1.0 --model_type=unigram --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/TextSummarization/tokenizer_temp_corpus.txt
  input_format: 
  model_prefix: /kaggle/working/TextSummarization/tokenizer
  model_type: UNIGRAM
  vocab_size: 30000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  sp

SentencePiece tokenizer trained and saved as /kaggle/working/TextSummarization/tokenizer.model
Tokenizing data from /kaggle/working/TextSummarization/data/processed/train_normalized.csv...


Tokenizing and padding: 100%|██████████| 20000/20000 [00:18<00:00, 1071.58it/s]


Tokenized data saved to /kaggle/working/TextSummarization/data/processed/train_tokenized.pt
Tokenizing data from /kaggle/working/TextSummarization/data/processed/validation_normalized.csv...


Tokenizing and padding: 100%|██████████| 13368/13368 [00:12<00:00, 1104.83it/s]


Tokenized data saved to /kaggle/working/TextSummarization/data/processed/validation_tokenized.pt
Tokenizing data from /kaggle/working/TextSummarization/data/processed/test_normalized.csv...


Tokenizing and padding: 100%|██████████| 11490/11490 [00:10<00:00, 1088.40it/s]


Tokenized data saved to /kaggle/working/TextSummarization/data/processed/test_tokenized.pt
PAD ID: 0
BOS ID: 2
EOS ID: 3
VOCAB SIZE: 30000


In [10]:
import os
import pandas as pd
import spacy
import torch
import sentencepiece as spm
from tqdm import tqdm

# Define constants
MAX_LEN = 64
VOCAB_SIZE = 30000 # Should match tokenizer training
PAD_ID = 0 # SentencePiece default pad_id
BOS_ID = 2 # SentencePiece default bos_id
EOS_ID = 3 # SentencePiece default eos_id

# Define paths
BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
TOKENIZER_MODEL_PATH = f"{BASE_DIR}/tokenizer.model"
SPACY_MODEL = "en_core_web_sm" # Using a small model for faster processing

# Load spaCy model
try:
    nlp = spacy.load(SPACY_MODEL)
    print(f"spaCy model '{SPACY_MODEL}' loaded successfully.")
except OSError:
    print(f"Downloading spaCy model '{SPACY_MODEL}'...")
    os.system(f"python -m spacy download {SPACY_MODEL}")
    nlp = spacy.load(SPACY_MODEL)
    print(f"spaCy model '{SPACY_MODEL}' downloaded and loaded.")

def load_tokenizer(model_path: str):
    """
    Loads a pre-trained SentencePiece tokenizer.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Tokenizer model not found at {model_path}")
    sp = spm.SentencePieceProcessor()
    sp.load(model_path)
    return sp

def create_tag_mappings(nlp_model):
    """
    Creates integer mappings for POS, DEP, and NER tags from spaCy's vocabulary.
    This function was updated to resolve AttributeError: 'spacy.vocab.Vocab' object has no attribute 'entity'.
    It now correctly accesses NER labels via `nlp_model.get_pipe('ner').labels`.
    """
    # Get POS tags from the tagger component labels if available
    pos_labels = list(nlp_model.pipe_labels.get('tagger', []))
    # Get DEP tags from the parser component labels if available
    dep_labels = list(nlp_model.pipe_labels.get('parser', []))
    
    # Get NER labels from the 'ner' pipeline component
    ner_labels = []
    if 'ner' in nlp_model.pipe_names:
        ner_labels = list(nlp_model.get_pipe('ner').labels)

    # Create mappings, ensuring unique and sorted order for consistent IDs
    pos_map = {tag: i for i, tag in enumerate(sorted(list(set(pos_labels))))}
    dep_map = {tag: i for i, tag in enumerate(sorted(list(set(dep_labels))))}
    ner_map = {tag: i for i, tag in enumerate(sorted(list(set(ner_labels))))}

    return pos_map, dep_map, ner_map

def process_and_tokenize_data(sp_tokenizer, input_path: str, output_path: str, max_len: int,
                              pos_map: dict, dep_map: dict, ner_map: dict, ner_map_o_id: int): # Added ner_map_o_id
    """
    Processes 'article' and 'highlights' columns:
    1. Applies spaCy to extract POS, DEP, and NER tags.
    2. Tokenizes text using SentencePiece.
    3. Aligns spaCy features with SentencePiece tokens.
    4. Pads/truncates all sequences (text, POS, DEP, NER) to max_len.
    5. Saves the data as a PyTorch .pt file.
    """
    print(f"Processing and tokenizing data from {input_path}...")
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}. Skipping processing.")
        return
    except Exception as e:
        print(f"Error reading CSV {input_path}: {e}")
        return

    all_articles_ids = []
    all_highlights_ids = []
    all_articles_pos_ids = []
    all_highlights_pos_ids = []
    all_articles_dep_ids = []
    all_highlights_dep_ids = []
    all_articles_ner_ids = []
    all_highlights_ner_ids = []

    # Get default ID for unknown tags (e.g., if a tag is not in our map)
    # Using 0 for unknown, assuming PAD_ID is 0. Be careful if PAD_ID changes.
    UNK_TAG_ID = len(pos_map) if pos_map else 1 # Simple arbitrary ID beyond known tags
    UNK_NER_ID = len(ner_map) if ner_map else 1


    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing with spaCy & Tokenizing"):
        article_text = str(row['article'])
        highlights_text = str(row['highlights'])

        # Process article
        doc_article = nlp(article_text)
        article_sp_tokens = sp_tokenizer.encode_as_pieces(article_text)
        article_sp_ids = sp_tokenizer.encode_as_ids(article_text)

        current_article_pos_ids = []
        current_article_dep_ids = []
        current_article_ner_ids = []

        # Simple alignment: For each SentencePiece token, find the first spaCy token
        # it corresponds to and take its features. This is a heuristic.
        spacy_token_idx = 0
        for sp_token in article_sp_tokens:
            if spacy_token_idx < len(doc_article):
                # Try to find a matching spaCy token for the current SentencePiece token
                # This alignment is an approximation. For subword tokens, it's tricky.
                # Here, we just take the feature of the current spaCy token.
                current_spacy_token = doc_article[spacy_token_idx]
                current_article_pos_ids.append(pos_map.get(current_spacy_token.pos_, UNK_TAG_ID))
                current_article_dep_ids.append(dep_map.get(current_spacy_token.dep_, UNK_TAG_ID))

                # For NER, check if the token is part of an entity
                ner_tag_found = False
                for ent in doc_article.ents:
                    if current_spacy_token.idx >= ent.start_char and \
                       current_spacy_token.idx < ent.end_char: # check if token is within entity span
                        current_article_ner_ids.append(ner_map.get(ent.label_, UNK_NER_ID))
                        ner_tag_found = True
                        break
                if not ner_tag_found:
                    current_article_ner_ids.append(ner_map_o_id) # Used passed in ner_map_o_id

                # Advance spaCy token index if the current SentencePiece token fully "consumes" it.
                # This is a very rough heuristic; true alignment needs more complex logic.
                # For `max_len=64` and simple feature addition, this might suffice.
                if current_spacy_token.text in sp_token or sp_token in current_spacy_token.text:
                    # If SentencePiece token seems to relate to current spaCy token, consider moving on.
                    # This logic is imperfect for subwords but helps avoid infinite loops.
                    pass
                else:
                    spacy_token_idx += 1 # Advance if no clear match

            else:
                # If SentencePiece tokens are left but no more spaCy tokens
                current_article_pos_ids.append(PAD_ID) # Or UNK_TAG_ID
                current_article_dep_ids.append(PAD_ID) # Or UNK_TAG_ID
                current_article_ner_ids.append(PAD_ID) # Or UNK_NER_ID


        # Process highlights (similar logic)
        doc_highlights = nlp(highlights_text)
        highlights_sp_tokens = sp_tokenizer.encode_as_pieces(highlights_text)
        highlights_sp_ids = sp_tokenizer.encode_as_ids(highlights_text)

        current_highlights_pos_ids = []
        current_highlights_dep_ids = []
        current_highlights_ner_ids = []

        spacy_token_idx = 0
        for sp_token in highlights_sp_tokens:
            if spacy_token_idx < len(doc_highlights):
                current_spacy_token = doc_highlights[spacy_token_idx]
                current_highlights_pos_ids.append(pos_map.get(current_spacy_token.pos_, UNK_TAG_ID))
                current_highlights_dep_ids.append(dep_map.get(current_spacy_token.dep_, UNK_TAG_ID))

                ner_tag_found = False
                for ent in doc_highlights.ents:
                    if current_spacy_token.idx >= ent.start_char and \
                       current_spacy_token.idx < ent.end_char:
                        current_highlights_ner_ids.append(ner_map.get(ent.label_, UNK_NER_ID))
                        ner_tag_found = True
                        break
                if not ner_tag_found:
                    current_highlights_ner_ids.append(ner_map_o_id) # Used passed in ner_map_o_id

                if current_spacy_token.text in sp_token or sp_token in current_spacy_token.text:
                    pass
                else:
                    spacy_token_idx += 1

            else:
                current_highlights_pos_ids.append(PAD_ID)
                current_highlights_dep_ids.append(PAD_ID)
                current_highlights_ner_ids.append(PAD_ID)


        # Helper function to process and pad/truncate features
        def get_features(text, sp_tok, max_len, nlp_model, pos_m, dep_m, ner_m, unk_pos_id, unk_dep_id, unk_ner_id_o_local): # Renamed ner_map_o_id to unk_ner_id_o_local to avoid shadowing
            doc = nlp_model(text)
            text_ids = sp_tok.encode_as_ids(text) # SentencePiece IDs
            pos_ids = []
            dep_ids = []
            ner_ids = []

            # Map spaCy tokens' features to SentencePiece tokens.
            # This is a heuristic. For each spaCy token, take its features.
            # Then, append these features for each SentencePiece token derived from this spaCy token's text.
            # A more robust alignment would be complex.
            # For this context, we will simply process spaCy tokens and map them to their corresponding
            # SentencePiece tokens. If a spaCy token leads to multiple SP tokens, duplicate features.
            # If a spaCy token is split from a word, the alignment is a compromise.

            sp_pieces_raw = sp_tok.encode_as_pieces(text)
            sp_ids_raw = sp_tok.encode_as_ids(text)

            current_char_idx = 0
            sp_idx = 0
            for spacy_token in doc:
                # Find which SentencePiece tokens correspond to this spaCy token
                spacy_token_text = spacy_token.text
                
                # Check for overlap with SentencePiece tokens
                temp_sp_idx = sp_idx
                while temp_sp_idx < len(sp_pieces_raw):
                    sp_piece = sp_pieces_raw[temp_sp_idx]
                    if sp_piece in spacy_token_text or spacy_token_text in sp_piece:
                        # This SentencePiece token seems to relate to the current spaCy token
                        pos_ids.append(pos_m.get(spacy_token.pos_, unk_pos_id))
                        dep_ids.append(dep_m.get(spacy_token.dep_, unk_dep_id))
                        
                        ner_tag_found = False
                        for ent in doc.ents:
                            if spacy_token.i >= ent.start and spacy_token.i < ent.end: # Check index within entity
                                ner_ids.append(ner_m.get(ent.label_, unk_ner_id_o_local)) # Used passed in unk_ner_id_o_local
                                ner_tag_found = True
                                break
                        if not ner_tag_found:
                            ner_ids.append(unk_ner_id_o_local) # 'O' for Outside entity
                        temp_sp_idx += 1
                    else:
                        break # No more SentencePiece tokens related to this spaCy token
                sp_idx = temp_sp_idx
            
            # If SentencePiece has more tokens than spaCy provided mapped features (due to
            # short spaCy doc or complex alignment), fill remaining with UNK/PAD.
            while len(pos_ids) < len(sp_ids_raw):
                pos_ids.append(unk_pos_id)
                dep_ids.append(unk_dep_id)
                ner_ids.append(unk_ner_id_o_local)

            # Pad and truncate
            # Ensure text_ids, pos_ids, dep_ids, ner_ids all have same length after EOS
            
            text_ids = text_ids[:max_len-1] + [EOS_ID]
            pos_ids = pos_ids[:max_len-1] + [PAD_ID] # Placeholder for EOS
            dep_ids = dep_ids[:max_len-1] + [PAD_ID] # Placeholder for EOS
            ner_ids = ner_ids[:max_len-1] + [PAD_ID] # Placeholder for EOS

            text_ids = text_ids + [PAD_ID] * (max_len - len(text_ids))
            pos_ids = pos_ids + [PAD_ID] * (max_len - len(pos_ids))
            dep_ids = dep_ids + [PAD_ID] * (max_len - len(dep_ids))
            ner_ids = ner_ids + [PAD_ID] * (max_len - len(ner_ids))

            return text_ids[:max_len], pos_ids[:max_len], dep_ids[:max_len], ner_ids[:max_len]


        # Process article and highlights
        article_sp_ids, article_pos_ids, article_dep_ids, article_ner_ids = get_features(
            article_text, sp_tokenizer, max_len, nlp, pos_map, dep_map, ner_map, UNK_TAG_ID, UNK_TAG_ID, ner_map_o_id # Pass ner_map_o_id
        )
        highlights_sp_ids, highlights_pos_ids, highlights_dep_ids, highlights_ner_ids = get_features(
            highlights_text, sp_tokenizer, max_len, nlp, pos_map, dep_map, ner_map, UNK_TAG_ID, UNK_TAG_ID, ner_map_o_id # Pass ner_map_o_id
        )

        all_articles_ids.append(article_sp_ids)
        all_highlights_ids.append(highlights_sp_ids)
        all_articles_pos_ids.append(article_pos_ids)
        all_highlights_pos_ids.append(highlights_pos_ids)
        all_articles_dep_ids.append(article_dep_ids)
        all_highlights_dep_ids.append(highlights_dep_ids)
        all_articles_ner_ids.append(article_ner_ids)
        all_highlights_ner_ids.append(highlights_ner_ids)

    # Convert to PyTorch tensors
    processed_data = {
        'articles_ids': torch.tensor(all_articles_ids, dtype=torch.long),
        'highlights_ids': torch.tensor(all_highlights_ids, dtype=torch.long),
        'articles_pos_ids': torch.tensor(all_articles_pos_ids, dtype=torch.long),
        'highlights_pos_ids': torch.tensor(all_highlights_pos_ids, dtype=torch.long),
        'articles_dep_ids': torch.tensor(all_articles_dep_ids, dtype=torch.long),
        'highlights_dep_ids': torch.tensor(all_highlights_dep_ids, dtype=torch.long),
        'articles_ner_ids': torch.tensor(all_articles_ner_ids, dtype=torch.long),
        'highlights_ner_ids': torch.tensor(all_highlights_ner_ids, dtype=torch.long),
        'pad_id': PAD_ID,
        'bos_id': BOS_ID,
        'eos_id': EOS_ID,
        'vocab_size': sp_tokenizer.get_piece_size(),
        'pos_vocab_size': len(pos_map) + 1, # +1 for UNK
        'dep_vocab_size': len(dep_map) + 1, # +1 for UNK
        'ner_vocab_size': len(ner_map) + 1  # +1 for 'O' / UNK
    }
    try:
        torch.save(processed_data, output_path)
        print(f"Processed data with spaCy features saved to {output_path}")
    except Exception as e:
        print(f"Error saving processed data to {output_path}: {e}")

if __name__ == "__main__":
    # Ensure directories exist
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    # Load tokenizer
    sp_tokenizer = load_tokenizer(TOKENIZER_MODEL_PATH)

    # Create tag mappings
    pos_map, dep_map, ner_map = create_tag_mappings(nlp)
    # Add 'O' (Outside) to NER map if it's not already there and define its ID
    if 'O' not in ner_map:
        ner_map['O'] = len(ner_map) # Assign a new ID for 'O'
    NER_MAP_O_ID = ner_map['O'] # Get the ID for 'O'

    # Paths to normalized data
    train_normalized_path = f"{PROCESSED_DATA_DIR}/train_normalized.csv"
    validation_normalized_path = f"{PROCESSED_DATA_DIR}/validation_normalized.csv"
    test_normalized_path = f"{PROCESSED_DATA_DIR}/test_normalized.csv"

    # Process each dataset
    process_and_tokenize_data(sp_tokenizer, train_normalized_path,
                              f"{PROCESSED_DATA_DIR}/train_processed.pt", MAX_LEN,
                              pos_map, dep_map, ner_map, NER_MAP_O_ID) # Passed NER_MAP_O_ID
    process_and_tokenize_data(sp_tokenizer, validation_normalized_path,
                              f"{PROCESSED_DATA_DIR}/validation_processed.pt", MAX_LEN,
                              pos_map, dep_map, ner_map, NER_MAP_O_ID) # Passed NER_MAP_O_ID
    process_and_tokenize_data(sp_tokenizer, test_normalized_path,
                              f"{PROCESSED_DATA_DIR}/test_processed.pt", MAX_LEN,
                              pos_map, dep_map, ner_map, NER_MAP_O_ID) # Passed NER_MAP_O_ID

    print("\nPreprocessing complete. Data is saved in .pt files with token IDs and spaCy features.")


spaCy model 'en_core_web_sm' loaded successfully.
Processing and tokenizing data from /kaggle/working/TextSummarization/data/processed/train_normalized.csv...


Processing with spaCy & Tokenizing: 100%|██████████| 20000/20000 [1:17:35<00:00,  4.30it/s]


Processed data with spaCy features saved to /kaggle/working/TextSummarization/data/processed/train_processed.pt
Processing and tokenizing data from /kaggle/working/TextSummarization/data/processed/validation_normalized.csv...


Processing with spaCy & Tokenizing: 100%|██████████| 13368/13368 [50:30<00:00,  4.41it/s] 


Processed data with spaCy features saved to /kaggle/working/TextSummarization/data/processed/validation_processed.pt
Processing and tokenizing data from /kaggle/working/TextSummarization/data/processed/test_normalized.csv...


Processing with spaCy & Tokenizing: 100%|██████████| 11490/11490 [43:24<00:00,  4.41it/s] 


Processed data with spaCy features saved to /kaggle/working/TextSummarization/data/processed/test_processed.pt

Preprocessing complete. Data is saved in .pt files with token IDs and spaCy features.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 n_layers: int = 1, dropout: float = 0.5, pad_idx: int = 0):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers,
                           bidirectional=True, dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)

        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor, src_len: torch.Tensor):
        embedded = self.dropout(self.embedding(src))

        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, src_len.cpu(), batch_first=True, enforce_sorted=False)

        packed_outputs, (hidden, cell) = self.rnn(packed_embedded)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True, total_length=src.shape[1])

        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)))
        cell = torch.tanh(self.fc_cell(torch.cat((cell[-2, :, :], cell[-1, :, :]), dim=1)))
        hidden = hidden.unsqueeze(0).repeat(self.n_layers, 1, 1)
        cell = cell.unsqueeze(0).repeat(self.n_layers, 1, 1)

        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.attn_hidden = nn.Linear(hidden_dim, hidden_dim)

        self.attn_encoder_outputs = nn.Linear(hidden_dim * 2, hidden_dim)

        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor = None):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1)

        energy = torch.tanh(self.attn_hidden(hidden) + self.attn_encoder_outputs(encoder_outputs))

        attention_scores = self.v(energy).squeeze(2)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e10)

        attention_weights = F.softmax(attention_scores, dim=1)

        context_vector = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context_vector, attention_weights


class Decoder(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 n_layers: int = 1, dropout: float = 0.5, pad_idx: int = 0):
        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)

        self.attention = Attention(hidden_dim)

        self.rnn = nn.LSTM(embedding_dim + (hidden_dim * 2), hidden_dim,
                           num_layers=n_layers, dropout=dropout if n_layers > 1 else 0,
                           batch_first=True)

        self.fc_out = nn.Linear(hidden_dim + (hidden_dim * 2) + embedding_dim, vocab_size)


    def forward(self, input_token: torch.Tensor, hidden: torch.Tensor,
                cell: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor):
        embedded = self.dropout(self.embedding(input_token))

        attn_hidden_for_attention = hidden[-1, :, :]

        context_vector, attention_weights = self.attention(attn_hidden_for_attention, encoder_outputs, mask)

        rnn_input = torch.cat((embedded, context_vector.unsqueeze(1)), dim=2)

        output_rnn, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        prediction_input = torch.cat((output_rnn.squeeze(1),
                                      context_vector,
                                      embedded.squeeze(1)),
                                     dim=1)

        output = self.fc_out(prediction_input)

        return output, hidden, cell, attention_weights


class Seq2Seq(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, device: torch.device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hidden_dim == decoder.hidden_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Number of layers in encoder and decoder must be equal!"

    def forward(self, src: torch.Tensor, src_len: torch.Tensor, trg: torch.Tensor, trg_len: torch.Tensor, teacher_forcing_ratio: float = 0.5):
        batch_size = src.shape[0]
        trg_seq_len = trg.shape[1]
        vocab_size = self.decoder.vocab_size

        encoder_outputs, hidden, cell = self.encoder(src, src_len)

        mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
        for j, length in enumerate(src_len):
            mask[j, :length] = True

        outputs = torch.zeros(batch_size, trg_seq_len - 1, vocab_size).to(self.device)

        input_token = trg[:, 0].unsqueeze(1)

        for t in range(1, trg_seq_len):
            output, hidden, cell, _ = self.decoder(input_token, hidden, cell, encoder_outputs, mask)

            outputs[:, t-1, :] = output

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input_token = trg[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)

        return outputs



In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd


MAX_LEN = 64
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 1
DROPOUT = 0.5
BATCH_SIZE = 32
NUM_EPOCHS = 10
CLIP_GRADIENT = 1.0

BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDataset(Dataset):
    def __init__(self, data_path: str):
        print(f"Loading data from {data_path}...")
        try:
            data = torch.load(data_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Processed data file not found at {data_path}")
        except Exception as e:
            raise Exception(f"Error loading data from {data_path}: {e}")

        self.articles = data['articles_ids']
        self.highlights = data['highlights_ids']
        self.pad_id = data['pad_id']
        self.bos_id = data['bos_id']
        self.eos_id = data['eos_id']
        self.vocab_size = data['vocab_size']

        self.article_lengths = self._calculate_lengths(self.articles, self.pad_id)
        self.highlight_lengths = self._calculate_lengths(self.highlights, self.eos_id, include_eos=True)


        print(f"Loaded {len(self.articles)} samples.")
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"PAD ID: {self.pad_id}, BOS ID: {self.bos_id}, EOS ID: {self.eos_id}")

    def _calculate_lengths(self, sequences: torch.Tensor, stop_id: int, include_eos: bool = False) -> torch.Tensor:
        lengths = []
        for seq in sequences:
            indices = (seq == stop_id).nonzero(as_tuple=True)[0]
            if len(indices) > 0:
                length = indices[0].item() + (1 if include_eos else 0)
            else:
                length = len(seq)
            lengths.append(min(length, MAX_LEN))
        return torch.tensor(lengths, dtype=torch.long)


    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        return {
            'article': self.articles[idx],
            'highlight': self.highlights[idx],
            'article_len': self.article_lengths[idx],
            'highlight_len': self.highlight_lengths[idx]
        }

def train_model(model, dataloader, optimizer, criterion, clip_gradient, device):
    model.train()
    epoch_loss = 0
    pbar = tqdm(dataloader, desc="Training")

    for i, batch in enumerate(pbar):
        src = batch['article'].to(device)
        trg = batch['highlight'].to(device)
        src_len = batch['article_len'].to(device)
        trg_len = batch['highlight_len'].to(device)

        optimizer.zero_grad()

        output = model(src, src_len, trg, trg_len)

        loss = criterion(output.reshape(-1, output.shape[-1]), trg[:, 1:].reshape(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_gradient)

        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return epoch_loss / len(dataloader)

def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validation")
        for i, batch in enumerate(pbar):
            src = batch['article'].to(device)
            trg = batch['highlight'].to(device)
            src_len = batch['article_len'].to(device)
            trg_len = batch['highlight_len'].to(device)

            output = model(src, src_len, trg, trg_len, 0.0) # No teacher forcing during evaluation

            loss = criterion(output.reshape(-1, output.shape[-1]), trg[:, 1:].reshape(-1))

            epoch_loss += loss.item()
            pbar.set_postfix(val_loss=loss.item())

    return epoch_loss / len(dataloader)


if __name__ == "__main__":
    train_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/train_processed.pt")
    val_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/validation_processed.pt")

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    VOCAB_SIZE = train_dataset.vocab_size
    PAD_ID = train_dataset.pad_id

    encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID)
    decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID)
    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

    def init_weights(m):
        for name, param in m.named_parameters():
            if 'weight' in name:
                nn.init.normal_(param.data, mean=0, std=0.01)
            else:
                nn.init.constant_(param.data, 0)
    model.apply(init_weights)

    print(f"Number of parameters in model: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    optimizer = optim.Adam(model.parameters(), lr=3e-4)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID).to(DEVICE)

    best_val_loss = float('inf')

    print("\nStarting Supervised Training...")
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

        train_loss = train_model(model, train_dataloader, optimizer, criterion, CLIP_GRADIENT, DEVICE)
        val_loss = evaluate_model(model, val_dataloader, criterion, DEVICE)

        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'vocab_size': VOCAB_SIZE,
                'pad_id': PAD_ID,
                'bos_id': train_dataset.bos_id,
                'eos_id': train_dataset.eos_id,
                'max_len': MAX_LEN,
                'embedding_dim': EMBEDDING_DIM,
                'hidden_dim': HIDDEN_DIM,
                'n_layers': N_LAYERS,
                'dropout': DROPOUT
            }, f"{CHECKPOINTS_DIR}/supervised_best_model.pt")
            print(f"  Saved best model with validation loss: {best_val_loss:.4f}")

    print("\nSupervised training complete!")


Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/train_processed.pt...
Loaded 20000 samples.
Vocabulary size: 30000
PAD ID: 0, BOS ID: 2, EOS ID: 3
Loading data from /kaggle/working/TextSummarization/data/processed/validation_processed.pt...
Loaded 13368 samples.
Vocabulary size: 30000
PAD ID: 0, BOS ID: 2, EOS ID: 3
Number of parameters in model: 77815600

Starting Supervised Training...

Epoch 1/10


Validation: 100%|██████████| 418/418 [02:36<00:00,  2.67it/s, val_loss=7.56]


  Train Loss: 7.6916
  Val Loss: 7.5647
  Saved best model with validation loss: 7.5647

Epoch 2/10


Validation: 100%|██████████| 418/418 [02:36<00:00,  2.67it/s, val_loss=7.5] 


  Train Loss: 7.3518
  Val Loss: 7.4746
  Saved best model with validation loss: 7.4746

Epoch 3/10


...

  Train Loss: 6.5392
  Val Loss: 7.3300
  Saved best model with validation loss: 7.3300

Epoch 9/10


Validation: 100%|██████████| 418/418 [02:37<00:00,  2.65it/s, val_loss=7.43]


  Train Loss: 6.4285
  Val Loss: 7.3662

Epoch 10/10


Validation: 100%|██████████| 418/418 [02:37<00:00,  2.65it/s, val_loss=7.45]

  Train Loss: 6.3277
  Val Loss: 7.3639

Supervised training complete!


In [16]:
!pip install rouge_score bert_score textstat nltk --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.0 MB/s eta 0:00:00:00:0100:01


In [22]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import sentencepiece as spm
import json
import re

import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.corpus import wordnet
import textstat

try:
    nltk.data.find('corpora/wordnet')
except (LookupError, Exception):
    print("Downloading 'wordnet' NLTK data...")
    nltk.download('wordnet')
try:
    nltk.data.find('tokenizers/punkt')
except (LookupError, Exception):
    print("Downloading 'punkt' NLTK data...")
    nltk.download('punkt')

from rouge_score import rouge_scorer
from bert_score import score as bert_score_calc

MAX_LEN = 64
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 1
DROPOUT = 0.5
BATCH_SIZE_EVAL = 64

BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
RESULTS_DIR = f"{BASE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDataset(Dataset):
    def __init__(self, data_path: str):
        print(f"Loading data from {data_path} for evaluation...")
        try:
            data = torch.load(data_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Processed data file not found at {data_path}")
        except Exception as e:
            raise Exception(f"Error loading data from {data_path}: {e}")

        self.articles = data['articles_ids']
        self.highlights = data['highlights_ids']
        self.pad_id = data['pad_id']
        self.bos_id = data['bos_id']
        self.eos_id = data['eos_id']
        self.vocab_size = data['vocab_size']

        self.article_lengths = self._calculate_lengths(self.articles, self.pad_id)
        self.highlight_lengths = self._calculate_lengths(self.highlights, self.eos_id, include_eos=True)

        print(f"Loaded {len(self.articles)} samples for evaluation.")

    def _calculate_lengths(self, sequences: torch.Tensor, stop_id: int, include_eos: bool = False) -> torch.Tensor:
        lengths = []
        for seq in sequences:
            indices = (seq == stop_id).nonzero(as_tuple=True)[0]
            if len(indices) > 0:
                length = indices[0].item() + (1 if include_eos else 0)
            else:
                length = len(seq)
            lengths.append(min(length, MAX_LEN))
        return torch.tensor(lengths, dtype=torch.long)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        return {
            'article': self.articles[idx],
            'highlight': self.highlights[idx],
            'article_len': self.article_lengths[idx],
            'highlight_len': self.highlight_lengths[idx]
        }

class Generator(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, vocab_size: int,
                 bos_id: int, eos_id: int, max_len: int, device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.vocab_size = vocab_size
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.max_len = max_len
        self.device = device

    def generate(self, src: torch.Tensor, src_len: torch.Tensor) -> torch.Tensor:
        self.encoder.eval()
        self.decoder.eval()

        with torch.no_grad():
            batch_size = src.shape[0]

            encoder_outputs, hidden, cell = self.encoder(src, src_len)

            mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
            for j, length in enumerate(src_len):
                mask[j, :length] = True

            input_token = torch.full((batch_size, 1), self.bos_id, dtype=torch.long).to(self.device)

            generated_sequences = torch.zeros(batch_size, self.max_len, dtype=torch.long).to(self.device)

            for t in range(self.max_len):
                output, hidden, cell, _ = self.decoder(input_token, hidden, cell, encoder_outputs, mask)

                predicted_token = output.argmax(1)

                generated_sequences[:, t] = predicted_token
                input_token = predicted_token.unsqueeze(1)

                if (predicted_token == self.eos_id).all():
                    break

            return generated_sequences

def convert_ids_to_text(sp_tokenizer, token_ids: torch.Tensor, eos_id: int, pad_id: int) -> list[str]:
    texts = []
    for seq_ids in token_ids.tolist():
        tokens = []
        for tok_id in seq_ids:
            if tok_id == eos_id:
                break
            if tok_id == pad_id:
                continue
            tokens.append(sp_tokenizer.id_to_piece(tok_id))
        text = "".join(tokens).replace(" ", " ").strip()
        texts.append(text)
    return texts

def calculate_cohesion_jaccard(text: str) -> float:
    sentences = nltk.sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0

    scores = []
    for i in range(len(sentences) - 1):
        words1 = set(nltk.word_tokenize(sentences[i].lower()))
        words2 = set(nltk.word_tokenize(sentences[i+1].lower()))
        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))
        if union > 0:
            scores.append(intersection / union)
        else:
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0


def evaluate_model(generator: Generator, dataloader: DataLoader, sp_tokenizer,
                   pad_id: int, eos_id: int, model_type: str = "supervised"):
    generator.eval()
    all_predicted_summaries = []
    all_reference_summaries = []

    pbar = tqdm(dataloader, desc="Generating summaries")
    for batch in pbar:
        src = batch['article'].to(DEVICE)
        trg_ref = batch['highlight'].to(DEVICE)
        src_len = batch['article_len'].to(DEVICE)

        generated_ids = generator.generate(src, src_len)

        predicted_texts = convert_ids_to_text(sp_tokenizer, generated_ids, eos_id, pad_id)
        reference_texts = convert_ids_to_text(sp_tokenizer, trg_ref, eos_id, pad_id)

        all_predicted_summaries.extend(predicted_texts)
        all_reference_summaries.extend(reference_texts)

    print("\nCalculating ROUGE scores...")
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': {'fmeasure': 0, 'precision': 0, 'recall': 0},
                    'rouge2': {'fmeasure': 0, 'precision': 0, 'recall': 0},
                    'rougeL': {'fmeasure': 0, 'precision': 0, 'recall': 0}}

    for i in tqdm(range(len(all_predicted_summaries)), desc="Computing ROUGE per sample"):
        scores = scorer.score(all_reference_summaries[i], all_predicted_summaries[i])
        for metric in rouge_scores:
            rouge_scores[metric]['fmeasure'] += scores[metric].fmeasure
            rouge_scores[metric]['precision'] += scores[metric].precision
            rouge_scores[metric]['recall'] += scores[metric].recall

    for metric in rouge_scores:
        for score_type in rouge_scores[metric]:
            rouge_scores[metric][score_type] /= len(all_predicted_summaries)

    print("\nROUGE Scores:")
    for metric, scores in rouge_scores.items():
        print(f"  {metric.upper()}:")
        print(f"    F1: {scores['fmeasure']:.4f}")
        print(f"    Precision: {scores['precision']:.4f}")
        print(f"    Recall: {scores['recall']:.4f}")

    print("\nCalculating BERTScore...")
    P, R, F1 = bert_score_calc(all_predicted_summaries, all_reference_summaries,
                               lang="en", verbose=True, device=str(DEVICE))

    bert_score_results = {
        'precision': P.mean().item(),
        'recall': R.mean().item(),
        'f1': F1.mean().item()
    }
    print(f"\nBERTScore:")
    print(f"  Precision: {bert_score_results['precision']:.4f}")
    print(f"  Recall: {bert_score_results['recall']:.4f}")
    print(f"  F1: {bert_score_results['f1']:.4f}")

    print("\nCalculating METEOR score...")
    meteor_scores = []
    for i in tqdm(range(len(all_predicted_summaries)), desc="Computing METEOR per sample"):
        # Handle cases where generated summary might be empty
        if not all_predicted_summaries[i].strip():
            meteor_scores.append(0.0) # Assign 0 score for empty generated summaries
            continue
        reference_tokenized = [nltk.word_tokenize(all_reference_summaries[i])]
        hypothesis_tokenized = nltk.word_tokenize(all_predicted_summaries[i])
        score = meteor_score(reference_tokenized, hypothesis_tokenized)
        meteor_scores.append(score)
    avg_meteor_score = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0
    print(f"\nMETEOR Score: {avg_meteor_score:.4f}")


    print("\nCalculating Readability and Cohesion scores...")
    readability_scores = {
        'flesch_reading_ease': [],
        'flesch_kincaid_grade': [],
        'dale_chall_readability_score': [],
        'automated_readability_index': [],
        'coleman_liau_index': [],
        'linsear_write_formula': [],
        'gunning_fog_score': [],
        'smog_index': []
    }
    cohesion_scores = []

    for summary_text in tqdm(all_predicted_summaries, desc="Computing Readability & Cohesion"):
        if summary_text and len(summary_text) > 0:
            readability_scores['flesch_reading_ease'].append(textstat.flesch_reading_ease(summary_text))
            readability_scores['flesch_kincaid_grade'].append(textstat.flesch_kincaid_grade(summary_text))
            readability_scores['dale_chall_readability_score'].append(textstat.dale_chall_readability_score(summary_text))
            readability_scores['automated_readability_index'].append(textstat.automated_readability_index(summary_text))
            readability_scores['coleman_liau_index'].append(textstat.coleman_liau_index(summary_text))
            readability_scores['linsear_write_formula'].append(textstat.linsear_write_formula(summary_text))
            readability_scores['gunning_fog_score'].append(textstat.gunning_fog(summary_text)) # Corrected function name
            readability_scores['smog_index'].append(textstat.smog_index(summary_text))
        else:
            for key in readability_scores:
                readability_scores[key].append(0.0)

        cohesion_scores.append(calculate_cohesion_jaccard(summary_text))

    avg_readability_scores = {k: sum(v) / len(v) if v else 0.0 for k, v in readability_scores.items()}
    avg_cohesion_score = sum(cohesion_scores) / len(cohesion_scores) if cohesion_scores else 0.0

    print("\nReadability Scores (Average):")
    for metric, score in avg_readability_scores.items():
        print(f"  {metric.replace('_', ' ').title()}: {score:.4f}")
    print(f"\nCohesion Score (Jaccard Similarity Average): {avg_cohesion_score:.4f}")


    evaluation_summary = {
        'model_type': model_type,
        'rouge_scores': rouge_scores,
        'bert_score': bert_score_results,
        'meteor_score': avg_meteor_score,
        'readability_scores': avg_readability_scores,
        'cohesion_score_jaccard': avg_cohesion_score,
        'num_samples_evaluated': len(all_predicted_summaries)
    }

    output_filename = f"evaluation_scores_{model_type}.json"
    predicted_filename = f"predicted_summaries_{model_type}.txt"

    with open(f"{RESULTS_DIR}/{predicted_filename}", "w", encoding="utf-8") as f:
        for pred in all_predicted_summaries:
            f.write(pred + "\n")
    print(f"Predicted summaries saved to {RESULTS_DIR}/{predicted_filename}")

    with open(f"{RESULTS_DIR}/{output_filename}", "w", encoding="utf-8") as f:
        json.dump(evaluation_summary, f, indent=4)
    print(f"Evaluation scores saved to {RESULTS_DIR}/{output_filename}")


if __name__ == "__main__":
    test_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/test_processed.pt")
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False)

    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer.load(f"{BASE_DIR}/tokenizer.model")

    supervised_best_model_path = f"{CHECKPOINTS_DIR}/supervised_best_model.pt"
    
    if not os.path.exists(supervised_best_model_path):
        print(f"Error: Supervised best model checkpoint not found at {supervised_best_model_path}.")
        print("Please run train_supervised.py first to train the base model.")
        exit()

    print(f"Loading supervised best model from {supervised_best_model_path} for evaluation...")
    checkpoint = torch.load(supervised_best_model_path, map_location=DEVICE)
    print(f"Loaded supervised model from epoch {checkpoint['epoch']+1}.")


    VOCAB_SIZE = checkpoint['vocab_size']
    PAD_ID = checkpoint['pad_id']
    BOS_ID = checkpoint['bos_id']
    EOS_ID = checkpoint['eos_id']
    MAX_LEN_CONST = checkpoint['max_len']
    EMBEDDING_DIM = checkpoint['embedding_dim']
    HIDDEN_DIM = checkpoint['hidden_dim']
    N_LAYERS = checkpoint['n_layers']
    DROPOUT = checkpoint['dropout']

    encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID).to(DEVICE)
    decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID).to(DEVICE)

    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    generator = Generator(model.encoder, model.decoder, VOCAB_SIZE, BOS_ID, EOS_ID, MAX_LEN_CONST, DEVICE).to(DEVICE)

    print("\nStarting evaluation of the supervised model...")
    evaluate_model(generator, test_dataloader, sp_tokenizer, PAD_ID, EOS_ID, model_type="supervised")
    print("\nEvaluation complete for the supervised model!")

Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/test_processed.pt for evaluation...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Loaded 11490 samples for evaluation.
Loading supervised best model from /kaggle/working/TextSummarization/checkpoints/supervised_best_model.pt for evaluation...
Loaded supervised model from epoch 8.

Starting evaluation of the supervised model...


Generating summaries: 100%|██████████| 180/180 [01:08<00:00,  2.63it/s]



Calculating ROUGE scores...


...

done in 630.96 seconds, 18.21 sentences/sec

BERTScore:
  Precision: 0.8639
  Recall: 0.8371
  F1: 0.8502

Calculating METEOR score...


Computing METEOR per sample: 100%|██████████| 11490/11490 [00:04<00:00, 2539.95it/s]



METEOR Score: 0.0000

Calculating Readability and Cohesion scores...


Computing Readability & Cohesion: 100%|██████████| 11490/11490 [00:05<00:00, 2011.48it/s]


Readability Scores (Average):
  Flesch Reading Ease: -336.7318
  Flesch Kincaid Grade: 60.4751
  Dale Chall Readability Score: 19.4761
  Automated Readability Index: 843.8342
  Coleman Liau Index: 750.4509
  Linsear Write Formula: 0.1646
  Gunning Fog Score: 26.9831
  Smog Index: 6.9257

Cohesion Score (Jaccard Similarity Average): 0.0000
Predicted summaries saved to /kaggle/working/TextSummarization/results/predicted_summaries_supervised.txt
Evaluation scores saved to /kaggle/working/TextSummarization/results/evaluation_scores_supervised.json

Evaluation complete for the supervised model!


In [25]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import glob
from torch.distributions import Categorical

from rouge_score import rouge_scorer

MAX_LEN = 64
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 1
DROPOUT = 0.5
BATCH_SIZE = 16
NUM_EPOCHS_RL = 5
CLIP_GRADIENT = 1.0

BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDataset(Dataset):
    def __init__(self, data_path: str):
        print(f"Loading data from {data_path} for RL...")
        try:
            data = torch.load(data_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Processed data file not found at {data_path}")
        except Exception as e:
            raise Exception(f"Error loading data from {data_path}: {e}")

        self.articles = data['articles_ids']
        self.highlights = data['highlights_ids']
        self.pad_id = data['pad_id']
        self.bos_id = data['bos_id']
        self.eos_id = data['eos_id']
        self.vocab_size = data['vocab_size']

        self.article_lengths = self._calculate_lengths(self.articles, self.pad_id)
        self.highlight_lengths = self._calculate_lengths(self.highlights, self.eos_id, include_eos=True)


        print(f"Loaded {len(self.articles)} samples for RL.")
        print(f"PAD ID: {self.pad_id}, BOS ID: {self.bos_id}, EOS ID: {self.eos_id}")

    def _calculate_lengths(self, sequences: torch.Tensor, stop_id: int, include_eos: bool = False) -> torch.Tensor:
        lengths = []
        for seq in sequences:
            indices = (seq == stop_id).nonzero(as_tuple=True)[0]
            if len(indices) > 0:
                length = indices[0].item() + (1 if include_eos else 0)
            else:
                length = len(seq)
            lengths.append(min(length, MAX_LEN))
        return torch.tensor(lengths, dtype=torch.long)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        return {
            'article': self.articles[idx],
            'highlight': self.highlights[idx],
            'article_len': self.article_lengths[idx],
            'highlight_len': self.highlight_lengths[idx]
        }

class RLAgent(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, vocab_size: int,
                 bos_id: int, eos_id: int, max_len: int, device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.vocab_size = vocab_size
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.max_len = max_len
        self.device = device

    def forward(self, src: torch.Tensor, src_len: torch.Tensor, trg_true: torch.Tensor = None):
        batch_size = src.shape[0]

        encoder_outputs, hidden, cell = self.encoder(src, src_len)

        mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
        for j, length in enumerate(src_len):
            mask[j, :length] = True

        input_token = torch.full((batch_size, 1), self.bos_id, dtype=torch.long).to(self.device)

        generated_sequences = torch.zeros(batch_size, self.max_len, dtype=torch.long).to(self.device)
        log_probs_list = []

        for t in range(self.max_len):
            output, hidden, cell, _ = self.decoder(input_token, hidden, cell, encoder_outputs, mask)

            probs = F.softmax(output, dim=-1)
            dist = Categorical(probs)
            sampled_token = dist.sample()

            log_prob = dist.log_prob(sampled_token)
            log_probs_list.append(log_prob)

            generated_sequences[:, t] = sampled_token
            input_token = sampled_token.unsqueeze(1)

            if (sampled_token == self.eos_id).all() and t > 0:
                break

        log_probs = torch.stack(log_probs_list, dim=1)

        return generated_sequences, log_probs


class RLEnvironment:
    def __init__(self, sp_tokenizer, reward_type: str = 'rougeL'):
        self.sp_tokenizer = sp_tokenizer
        self.scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.reward_type = reward_type
        if self.reward_type == 'bertscore':
            print("BERTScore is requested but commented out due to potential issues in Kaggle/offline environment and slowness. Using ROUGE-L.")
            self.reward_type = 'rougeL'


    def calculate_reward(self, generated_ids: torch.Tensor, reference_ids: torch.Tensor,
                         pad_id: int, eos_id: int) -> torch.Tensor:
        rewards = []
        generated_texts = []
        reference_texts = []

        for i in range(generated_ids.shape[0]):
            gen_tokens = []
            for token_id in generated_ids[i].tolist():
                if token_id == eos_id:
                    break
                if token_id != pad_id:
                    gen_tokens.append(self.sp_tokenizer.id_to_piece(token_id))
            generated_texts.append("".join(gen_tokens).replace(" ", " ").strip())

            ref_tokens = []
            for token_id in reference_ids[i].tolist():
                if token_id == eos_id:
                    break
                if token_id != pad_id:
                    ref_tokens.append(self.sp_tokenizer.id_to_piece(token_id))
            reference_texts.append("".join(ref_tokens).replace(" ", " ").strip())


        for i in range(len(generated_texts)):
            generated_text = generated_texts[i]
            reference_text = reference_texts[i]

            if not generated_text.strip():
                rewards.append(0.0)
                continue

            if self.reward_type == 'rougeL':
                scores = self.scorer.score(reference_text, generated_text)
                reward = scores['rougeL'].fmeasure
                rewards.append(reward)
            else:
                rewards.append(0.0)

        return torch.tensor(rewards, dtype=torch.float, device=DEVICE)

def find_latest_checkpoint(checkpoint_dir: str, prefix: str = 'agent_epoch') -> str | None:
    list_of_files = glob.glob(f"{checkpoint_dir}/{prefix}*.pt")
    if not list_of_files:
        return None
    latest_file = max(list_of_files, key=os.path.getctime)
    return latest_file

def train_rl_model(agent: RLAgent, environment: RLEnvironment, dataloader: DataLoader,
                   optimizer: optim.Optimizer, clip_gradient: float,
                   pad_id: int, eos_id: int, num_epochs: int, start_epoch: int = 0):
    agent.train()
    print(f"\nStarting RL Training from epoch {start_epoch}...")

    for epoch in range(start_epoch, num_epochs):
        print(f"\nRL Epoch {epoch+1}/{num_epochs}")
        epoch_loss = 0
        pbar = tqdm(dataloader, desc=f"RL Epoch {epoch+1}")

        for i, batch in enumerate(pbar):
            src = batch['article'].to(DEVICE)
            trg_ref = batch['highlight'].to(DEVICE)
            src_len = batch['article_len'].to(DEVICE)

            optimizer.zero_grad()

            generated_sequences, log_probs = agent(src, src_len)

            rewards = environment.calculate_reward(generated_sequences, trg_ref, pad_id, eos_id)

            batch_size = src.shape[0] # Define batch_size here

            generated_lengths = torch.zeros(generated_sequences.shape[0], dtype=torch.long, device=DEVICE)
            for j in range(generated_sequences.shape[0]):
                eos_idx = (generated_sequences[j] == eos_id).nonzero(as_tuple=True)[0]
                if len(eos_idx) > 0:
                    generated_lengths[j] = eos_idx[0].item() + 1
                else:
                    generated_lengths[j] = MAX_LEN

            log_probs_mask = torch.zeros_like(log_probs).bool()
            for j in range(log_probs.shape[0]):
                log_probs_mask[j, :generated_lengths[j]] = True

            masked_log_probs = log_probs.masked_select(log_probs_mask)

            expanded_rewards = rewards.unsqueeze(1).expand_as(log_probs)
            masked_rewards = expanded_rewards.masked_select(log_probs_mask)

            loss = (-masked_log_probs * masked_rewards).sum() / batch_size

            loss.backward()

            torch.nn.utils.clip_grad_norm_(agent.encoder.parameters(), clip_gradient)
            torch.nn.utils.clip_grad_norm_(agent.decoder.parameters(), clip_gradient)

            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix(rl_loss=loss.item(), avg_reward=rewards.mean().item())

        avg_epoch_loss = epoch_loss / len(dataloader)
        print(f"  RL Epoch {epoch+1} Average Loss: {avg_epoch_loss:.4f}")

        checkpoint_path = f"{CHECKPOINTS_DIR}/agent_epoch{epoch+1}.pt"
        torch.save({
            'epoch': epoch + 1,
            'encoder_state_dict': agent.encoder.state_dict(),
            'decoder_state_dict': agent.decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_epoch_loss
        }, checkpoint_path)
        print(f"  Saved RL checkpoint to {checkpoint_path}")

    print("\nReinforcement learning training complete!")


if __name__ == "__main__":
    supervised_model_path = f"{CHECKPOINTS_DIR}/supervised_best_model.pt"
    if not os.path.exists(supervised_model_path):
        print(f"Error: Supervised model checkpoint not found at {supervised_model_path}.")
        print("Please run train_supervised.py first to train the base model.")
        exit()

    print(f"Loading supervised model from {supervised_model_path}...")
    checkpoint = torch.load(supervised_model_path, map_location=DEVICE)

    VOCAB_SIZE = checkpoint['vocab_size']
    PAD_ID = checkpoint['pad_id']
    BOS_ID = checkpoint['bos_id']
    EOS_ID = checkpoint['eos_id']
    MAX_LEN = checkpoint['max_len']
    EMBEDDING_DIM = checkpoint['embedding_dim']
    HIDDEN_DIM = checkpoint['hidden_dim']
    N_LAYERS = checkpoint['n_layers']
    DROPOUT = checkpoint['dropout']

    encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID).to(DEVICE)
    decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID).to(DEVICE)

    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])

    agent = RLAgent(model.encoder, model.decoder, VOCAB_SIZE, BOS_ID, EOS_ID, MAX_LEN, DEVICE).to(DEVICE)

    optimizer = optim.Adam(list(agent.encoder.parameters()) + list(agent.decoder.parameters()), lr=3e-4)

    latest_rl_checkpoint = find_latest_checkpoint(CHECKPOINTS_DIR, prefix='agent_epoch')
    start_epoch = 0
    if latest_rl_checkpoint:
        print(f"Resuming RL training from {latest_rl_checkpoint}...")
        rl_checkpoint = torch.load(latest_rl_checkpoint, map_location=DEVICE)
        agent.encoder.load_state_dict(rl_checkpoint['encoder_state_dict'])
        agent.decoder.load_state_dict(rl_checkpoint['decoder_state_dict'])
        optimizer.load_state_dict(rl_checkpoint['optimizer_state_dict'])
        start_epoch = rl_checkpoint['epoch']
        print(f"Resumed from epoch {start_epoch}.")
    else:
        print("No previous RL checkpoint found. Starting RL training from scratch.")

    train_dataset_rl = SummarizationDataset(f"{PROCESSED_DATA_DIR}/train_processed.pt")
    train_dataloader_rl = DataLoader(train_dataset_rl, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer.load(f"{BASE_DIR}/tokenizer.model")

    rl_environment = RLEnvironment(sp_tokenizer=sp_tokenizer, reward_type='rougeL')

    train_rl_model(agent, rl_environment, train_dataloader_rl, optimizer,
                   CLIP_GRADIENT, PAD_ID, EOS_ID, NUM_EPOCHS_RL, start_epoch)


Using device: cuda
Loading supervised model from /kaggle/working/TextSummarization/checkpoints/supervised_best_model.pt...
No previous RL checkpoint found. Starting RL training from scratch.
Loading data from /kaggle/working/TextSummarization/data/processed/train_processed.pt for RL...
Loaded 20000 samples for RL.
PAD ID: 0, BOS ID: 2, EOS ID: 3

Starting RL Training from epoch 0...

RL Epoch 1/5


RL Epoch 1: 100%|██████████| 1250/1250 [14:47<00:00,  1.41it/s, avg_reward=0.129, rl_loss=4.78] 


  RL Epoch 1 Average Loss: 7.6966
  Saved RL checkpoint to /kaggle/working/TextSummarization/checkpoints/agent_epoch1.pt

RL Epoch 2/5


RL Epoch 2: 100%|██████████| 1250/1250 [14:21<00:00,  1.45it/s, avg_reward=0.103, rl_loss=9.02] 


  RL Epoch 2 Average Loss: 5.5796
  Saved RL checkpoint to /kaggle/working/TextSummarization/checkpoints/agent_epoch2.pt

RL Epoch 3/5


...

  RL Epoch 3 Average Loss: 7.2785
  Saved RL checkpoint to /kaggle/working/TextSummarization/checkpoints/agent_epoch3.pt

RL Epoch 4/5


RL Epoch 4: 100%|██████████| 1250/1250 [14:46<00:00,  1.41it/s, avg_reward=0.0722, rl_loss=4.98]


  RL Epoch 4 Average Loss: 6.4865
  Saved RL checkpoint to /kaggle/working/TextSummarization/checkpoints/agent_epoch4.pt

RL Epoch 5/5


RL Epoch 5: 100%|██████████| 1250/1250 [15:06<00:00,  1.38it/s, avg_reward=0.173, rl_loss=9.94] 


  RL Epoch 5 Average Loss: 10.0073
  Saved RL checkpoint to /kaggle/working/TextSummarization/checkpoints/agent_epoch5.pt

Reinforcement learning training complete!


In [27]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import sentencepiece as spm
import json
import re
import glob # Import glob for finding latest checkpoint

import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.corpus import wordnet
import textstat

try:
    nltk.data.find('corpora/wordnet')
except (LookupError, Exception):
    print("Downloading 'wordnet' NLTK data...")
    nltk.download('wordnet')
try:
    nltk.data.find('tokenizers/punkt')
except (LookupError, Exception):
    print("Downloading 'punkt' NLTK data...")
    nltk.download('punkt')

from rouge_score import rouge_scorer
from bert_score import score as bert_score_calc
MAX_LEN = 64
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 1
DROPOUT = 0.5
BATCH_SIZE_EVAL = 64

BASE_DIR = "/kaggle/working/TextSummarization"
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
RESULTS_DIR = f"{BASE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDataset(Dataset):
    def __init__(self, data_path: str):
        print(f"Loading data from {data_path} for evaluation...")
        try:
            data = torch.load(data_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Processed data file not found at {data_path}")
        except Exception as e:
            raise Exception(f"Error loading data from {data_path}: {e}")

        self.articles = data['articles_ids']
        self.highlights = data['highlights_ids']
        self.pad_id = data['pad_id']
        self.bos_id = data['bos_id']
        self.eos_id = data['eos_id']
        self.vocab_size = data['vocab_size']

        self.article_lengths = self._calculate_lengths(self.articles, self.pad_id)
        self.highlight_lengths = self._calculate_lengths(self.highlights, self.eos_id, include_eos=True)

        print(f"Loaded {len(self.articles)} samples for evaluation.")

    def _calculate_lengths(self, sequences: torch.Tensor, stop_id: int, include_eos: bool = False) -> torch.Tensor:
        lengths = []
        for seq in sequences:
            indices = (seq == stop_id).nonzero(as_tuple=True)[0]
            if len(indices) > 0:
                length = indices[0].item() + (1 if include_eos else 0)
            else:
                length = len(seq)
            lengths.append(min(length, MAX_LEN))
        return torch.tensor(lengths, dtype=torch.long)

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        return {
            'article': self.articles[idx],
            'highlight': self.highlights[idx],
            'article_len': self.article_lengths[idx],
            'highlight_len': self.highlight_lengths[idx]
        }

class Generator(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, vocab_size: int,
                 bos_id: int, eos_id: int, max_len: int, device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.vocab_size = vocab_size
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.max_len = max_len
        self.device = device

    def generate(self, src: torch.Tensor, src_len: torch.Tensor) -> torch.Tensor:
        self.encoder.eval()
        self.decoder.eval()

        with torch.no_grad():
            batch_size = src.shape[0]

            encoder_outputs, hidden, cell = self.encoder(src, src_len)

            mask = torch.zeros(src.shape[0], src.shape[1]).bool().to(self.device)
            for j, length in enumerate(src_len):
                mask[j, :length] = True

            input_token = torch.full((batch_size, 1), self.bos_id, dtype=torch.long).to(self.device)

            generated_sequences = torch.zeros(batch_size, self.max_len, dtype=torch.long).to(self.device)

            for t in range(self.max_len):
                output, hidden, cell, _ = self.decoder(input_token, hidden, cell, encoder_outputs, mask)

                predicted_token = output.argmax(1)

                generated_sequences[:, t] = predicted_token
                input_token = predicted_token.unsqueeze(1)

                if (predicted_token == self.eos_id).all():
                    break

            return generated_sequences

def convert_ids_to_text(sp_tokenizer, token_ids: torch.Tensor, eos_id: int, pad_id: int) -> list[str]:
    texts = []
    for seq_ids in token_ids.tolist():
        tokens = []
        for tok_id in seq_ids:
            if tok_id == eos_id:
                break
            if tok_id == pad_id:
                continue
            tokens.append(sp_tokenizer.id_to_piece(tok_id))
        text = "".join(tokens).replace(" ", " ").strip()
        texts.append(text)
    return texts

def calculate_cohesion_jaccard(text: str) -> float:
    sentences = nltk.sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0

    scores = []
    for i in range(len(sentences) - 1):
        words1 = set(nltk.word_tokenize(sentences[i].lower()))
        words2 = set(nltk.word_tokenize(sentences[i+1].lower()))
        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))
        if union > 0:
            scores.append(intersection / union)
        else:
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0


def evaluate_model(generator: Generator, dataloader: DataLoader, sp_tokenizer,
                   pad_id: int, eos_id: int, model_type: str = "supervised"):
    generator.eval()
    all_predicted_summaries = []
    all_reference_summaries = []

    pbar = tqdm(dataloader, desc="Generating summaries")
    for batch in pbar:
        src = batch['article'].to(DEVICE)
        trg_ref = batch['highlight'].to(DEVICE)
        src_len = batch['article_len'].to(DEVICE)

        generated_ids = generator.generate(src, src_len)

        predicted_texts = convert_ids_to_text(sp_tokenizer, generated_ids, eos_id, pad_id)
        reference_texts = convert_ids_to_text(sp_tokenizer, trg_ref, eos_id, pad_id)

        all_predicted_summaries.extend(predicted_texts)
        all_reference_summaries.extend(reference_texts)

    print("\nCalculating ROUGE scores...")
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': {'fmeasure': 0, 'precision': 0, 'recall': 0},
                    'rouge2': {'fmeasure': 0, 'precision': 0, 'recall': 0},
                    'rougeL': {'fmeasure': 0, 'precision': 0, 'recall': 0}}

    for i in tqdm(range(len(all_predicted_summaries)), desc="Computing ROUGE per sample"):
        scores = scorer.score(all_reference_summaries[i], all_predicted_summaries[i])
        for metric in rouge_scores:
            rouge_scores[metric]['fmeasure'] += scores[metric].fmeasure
            rouge_scores[metric]['precision'] += scores[metric].precision
            rouge_scores[metric]['recall'] += scores[metric].recall

    for metric in rouge_scores:
        for score_type in rouge_scores[metric]:
            rouge_scores[metric][score_type] /= len(all_predicted_summaries)

    print("\nROUGE Scores:")
    for metric, scores in rouge_scores.items():
        print(f"  {metric.upper()}:")
        print(f"    F1: {scores['fmeasure']:.4f}")
        print(f"    Precision: {scores['precision']:.4f}")
        print(f"    Recall: {scores['recall']:.4f}")

    print("\nCalculating BERTScore...")
    P, R, F1 = bert_score_calc(all_predicted_summaries, all_reference_summaries,
                               lang="en", verbose=True, device=str(DEVICE))

    bert_score_results = {
        'precision': P.mean().item(),
        'recall': R.mean().item(),
        'f1': F1.mean().item()
    }
    print(f"\nBERTScore:")
    print(f"  Precision: {bert_score_results['precision']:.4f}")
    print(f"  Recall: {bert_score_results['recall']:.4f}")
    print(f"  F1: {bert_score_results['f1']:.4f}")

    print("\nCalculating METEOR score...")
    meteor_scores = []
    for i in tqdm(range(len(all_predicted_summaries)), desc="Computing METEOR per sample"):
        if not all_predicted_summaries[i].strip():
            meteor_scores.append(0.0)
            continue
        reference_tokenized = [nltk.word_tokenize(all_reference_summaries[i])]
        hypothesis_tokenized = nltk.word_tokenize(all_predicted_summaries[i])
        score = meteor_score(reference_tokenized, hypothesis_tokenized)
        meteor_scores.append(score)
    avg_meteor_score = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0
    print(f"\nMETEOR Score: {avg_meteor_score:.4f}")


    print("\nCalculating Readability and Cohesion scores...")
    readability_scores = {
        'flesch_reading_ease': [],
        'flesch_kincaid_grade': [],
        'dale_chall_readability_score': [],
        'automated_readability_index': [],
        'coleman_liau_index': [],
        'linsear_write_formula': [],
        'gunning_fog_score': [],
        'smog_index': []
    }
    cohesion_scores = []

    for summary_text in tqdm(all_predicted_summaries, desc="Computing Readability & Cohesion"):
        if summary_text and len(summary_text) > 0:
            readability_scores['flesch_reading_ease'].append(textstat.flesch_reading_ease(summary_text))
            readability_scores['flesch_kincaid_grade'].append(textstat.flesch_kincaid_grade(summary_text))
            readability_scores['dale_chall_readability_score'].append(textstat.dale_chall_readability_score(summary_text))
            readability_scores['automated_readability_index'].append(textstat.automated_readability_index(summary_text))
            readability_scores['coleman_liau_index'].append(textstat.coleman_liau_index(summary_text))
            readability_scores['linsear_write_formula'].append(textstat.linsear_write_formula(summary_text))
            readability_scores['gunning_fog_score'].append(textstat.gunning_fog(summary_text))
            readability_scores['smog_index'].append(textstat.smog_index(summary_text))
        else:
            for key in readability_scores:
                readability_scores[key].append(0.0)

        cohesion_scores.append(calculate_cohesion_jaccard(summary_text))

    avg_readability_scores = {k: sum(v) / len(v) if v else 0.0 for k, v in readability_scores.items()}
    avg_cohesion_score = sum(cohesion_scores) / len(cohesion_scores) if cohesion_scores else 0.0

    print("\nReadability Scores (Average):")
    for metric, score in avg_readability_scores.items():
        print(f"  {metric.replace('_', ' ').title()}: {score:.4f}")
    print(f"\nCohesion Score (Jaccard Similarity Average): {avg_cohesion_score:.4f}")


    evaluation_summary = {
        'model_type': model_type,
        'rouge_scores': rouge_scores,
        'bert_score': bert_score_results,
        'meteor_score': avg_meteor_score,
        'readability_scores': avg_readability_scores,
        'cohesion_score_jaccard': avg_cohesion_score,
        'num_samples_evaluated': len(all_predicted_summaries)
    }

    output_filename = f"evaluation_scores_{model_type}.json"
    predicted_filename = f"predicted_summaries_{model_type}.txt"

    with open(f"{RESULTS_DIR}/{predicted_filename}", "w", encoding="utf-8") as f:
        for pred in all_predicted_summaries:
            f.write(pred + "\n")
    print(f"Predicted summaries saved to {RESULTS_DIR}/{predicted_filename}")

    with open(f"{RESULTS_DIR}/{output_filename}", "w", encoding="utf-8") as f:
        json.dump(evaluation_summary, f, indent=4)
    print(f"Evaluation scores saved to {RESULTS_DIR}/{output_filename}")


if __name__ == "__main__":
    test_dataset = SummarizationDataset(f"{PROCESSED_DATA_DIR}/test_processed.pt")
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False)

    sp_tokenizer = spm.SentencePieceProcessor()
    sp_tokenizer.load(f"{BASE_DIR}/tokenizer.model")

    # First, load the base model configuration from the supervised checkpoint
    supervised_best_model_path = f"{CHECKPOINTS_DIR}/supervised_best_model.pt"
    if not os.path.exists(supervised_best_model_path):
        print(f"Error: Supervised best model checkpoint not found at {supervised_best_model_path}.")
        print("Please run train_supervised.py first to train the base model.")
        exit()

    print(f"Loading base model configuration from {supervised_best_model_path}...")
    base_checkpoint = torch.load(supervised_best_model_path, map_location=DEVICE)
    
    VOCAB_SIZE = base_checkpoint['vocab_size']
    PAD_ID = base_checkpoint['pad_id']
    BOS_ID = base_checkpoint['bos_id']
    EOS_ID = base_checkpoint['eos_id']
    MAX_LEN_CONST = base_checkpoint['max_len']
    EMBEDDING_DIM = base_checkpoint['embedding_dim']
    HIDDEN_DIM = base_checkpoint['hidden_dim']
    N_LAYERS = base_checkpoint['n_layers']
    DROPOUT = base_checkpoint['dropout']

    # Instantiate encoder and decoder with these parameters
    encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID).to(DEVICE)
    decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_ID).to(DEVICE)
    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

    # Now, determine which model to load for evaluation (RL or Supervised)
    latest_rl_checkpoint = None
    list_of_rl_checkpoints = glob.glob(f"{CHECKPOINTS_DIR}/agent_epoch*.pt")
    if list_of_rl_checkpoints:
        latest_rl_checkpoint = max(list_of_rl_checkpoints, key=os.path.getctime)

    if latest_rl_checkpoint:
        print(f"Loading RL model from {latest_rl_checkpoint} for evaluation...")
        checkpoint_to_load = torch.load(latest_rl_checkpoint, map_location=DEVICE)
        model_type_eval = "rl"
        # Load state_dicts for encoder and decoder from RL checkpoint
        encoder.load_state_dict(checkpoint_to_load['encoder_state_dict'])
        decoder.load_state_dict(checkpoint_to_load['decoder_state_dict'])
        print(f"Loaded RL model from epoch {checkpoint_to_load['epoch']}.")
    else:
        print(f"No RL checkpoint found. Loading supervised best model from {supervised_best_model_path} for evaluation...")
        checkpoint_to_load = base_checkpoint # Already loaded as base_checkpoint
        model_type_eval = "supervised"
        # Load state_dict for the entire model from supervised checkpoint
        model.load_state_dict(checkpoint_to_load['model_state_dict'])
        print(f"Loaded supervised model from epoch {checkpoint_to_load['epoch']+1}.")
    
    # The generator will use the encoder and decoder components of the 'model' object,
    # which have now been loaded with the correct weights (either supervised or RL).
    generator = Generator(model.encoder, model.decoder, VOCAB_SIZE, BOS_ID, EOS_ID, MAX_LEN_CONST, DEVICE).to(DEVICE)

    print(f"\nStarting evaluation of the {model_type_eval} model...")
    evaluate_model(generator, test_dataloader, sp_tokenizer, PAD_ID, EOS_ID, model_type=model_type_eval)
    print(f"\nEvaluation complete for the {model_type_eval} model!")


Using device: cuda
Loading data from /kaggle/working/TextSummarization/data/processed/test_processed.pt for evaluation...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Loaded 11490 samples for evaluation.
Loading base model configuration from /kaggle/working/TextSummarization/checkpoints/supervised_best_model.pt...
Loading RL model from /kaggle/working/TextSummarization/checkpoints/agent_epoch5.pt for evaluation...
Loaded RL model from epoch 5.

Starting evaluation of the rl model...


Generating summaries: 100%|██████████| 180/180 [01:14<00:00,  2.43it/s]



Calculating ROUGE scores...


...

done in 405.16 seconds, 28.36 sentences/sec

BERTScore:
  Precision: 0.8607
  Recall: 0.8371
  F1: 0.8487

Calculating METEOR score...


Computing METEOR per sample: 100%|██████████| 11490/11490 [00:04<00:00, 2685.48it/s]



METEOR Score: 0.0000

Calculating Readability and Cohesion scores...


Computing Readability & Cohesion: 100%|██████████| 11490/11490 [00:00<00:00, 30721.12it/s]


Readability Scores (Average):
  Flesch Reading Ease: -1582.8846
  Flesch Kincaid Grade: 234.2884
  Dale Chall Readability Score: 19.4761
  Automated Readability Index: 936.2933
  Coleman Liau Index: 778.0178
  Linsear Write Formula: 0.5000
  Gunning Fog Score: 40.4000
  Smog Index: 8.8418

Cohesion Score (Jaccard Similarity Average): 0.0000
Predicted summaries saved to /kaggle/working/TextSummarization/results/predicted_summaries_rl.txt
Evaluation scores saved to /kaggle/working/TextSummarization/results/evaluation_scores_rl.json

Evaluation complete for the rl model!


In [28]:
import torch
import torch.nn as nn
from transformers import BartForConditionalGeneration, AutoTokenizer

class BartSummarizer(nn.Module):
    """
    A text summarization model using a pre-trained BART architecture.

    This module wraps the Hugging Face BartForConditionalGeneration model,
    providing a clean interface for training and generation.

    Args:
        model_name (str): The name of the pre-trained BART model to load
                          from Hugging Face Transformers (e.g., 'sshleifer/distilbart-cnn-12-6').
                          This model is specifically fine-tuned for summarization.
        pad_token_id (int): The ID of the padding token for the tokenizer.
                            Used for attention masking during training.
    """
    def __init__(self, model_name: str = 'sshleifer/distilbart-cnn-12-6', pad_token_id: int = None):
        super().__init__()
        # Load the pre-trained BART model for conditional generation (sequence-to-sequence)
        # This model includes an encoder and a decoder.
        self.model = BartForConditionalGeneration.from_pretrained(model_name)
        
        # Store pad_token_id, which is crucial for handling padded inputs
        self.pad_token_id = pad_token_id

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: torch.Tensor = None):
        """
        Forward pass for the BART summarizer.

        Args:
            input_ids (torch.Tensor): Tokenized input sequences (batch_size, sequence_length).
            attention_mask (torch.Tensor): Attention mask for input_ids (batch_size, sequence_length),
                                           1 for real tokens, 0 for padding.
            labels (torch.Tensor, optional): Target sequences for supervised training (batch_size, target_sequence_length).
                                            These are passed to the model internally for loss calculation.
                                            If None, the model is in generation/inference mode.

        Returns:
            transformers.modeling_outputs.Seq2SeqLMOutput:
                A dataclass containing various outputs, including:
                - loss (torch.Tensor): If `labels` are provided, the masked language modeling loss.
                - logits (torch.Tensor): Logits for the next token prediction (batch_size, sequence_length, vocab_size).
        """
        # BART's forward method handles the entire sequence-to-sequence process,
        # including encoder, decoder, and attention.
        # When `labels` are provided, it also calculates the loss internally.
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs

    def generate(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, 
                 max_length: int, num_beams: int = 4, early_stopping: bool = True) -> torch.Tensor:
        """
        Generates summaries using the BART model.

        Args:
            input_ids (torch.Tensor): Tokenized input sequences (batch_size, sequence_length).
            attention_mask (torch.Tensor): Attention mask for input_ids (batch_size, sequence_length).
            max_length (int): Maximum length of the generated summary.
            num_beams (int): Number of beams for beam search decoding. Higher values improve quality
                             at the cost of speed.
            early_stopping (bool): Whether to stop generation once all beam hypotheses have finished.

        Returns:
            torch.Tensor: Generated token IDs (batch_size, generated_sequence_length).
        """
        # The generate method is highly optimized and performs beam search decoding by default.
        generated_ids = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=early_stopping,
            pad_token_id=self.pad_token_id # Use the correct pad token ID
        )
        return generated_ids

    def get_tokenizer(self, model_name: str = 'sshleifer/distilbart-cnn-12-6'):
        """
        Loads and returns the corresponding tokenizer for the BART model.
        This tokenizer should be used for all preprocessing steps.
        """
        return AutoTokenizer.from_pretrained(model_name)



In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = 'sshleifer/distilbart-cnn-12-6'
MAX_SOURCE_LEN = 64
MAX_TARGET_LEN = 64
BATCH_SIZE = 16
NUM_EPOCHS = 10
CLIP_GRADIENT = 1.0

BASE_DIR = "/kaggle/working/TextSummarization"
RAW_DATA_DIR = f"{BASE_DIR}/data/raw" # Assuming normalized data is in raw/ or processed/
PROCESSED_DATA_DIR = f"{BASE_DIR}/data/processed" # Where normalized data might be after normalize.py
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

class SummarizationDatasetBART(Dataset):
    def __init__(self, csv_path: str, tokenizer, max_source_len: int, max_target_len: int, sample_size: int = None):
        print(f"Loading and tokenizing data from {csv_path} for BART training...")
        try:
            df = pd.read_csv(csv_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"CSV file not found at {csv_path}")
        except Exception as e:
            raise Exception(f"Error reading CSV {csv_path}: {e}")

        if sample_size and len(df) > sample_size:
            df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
            print(f"  Sampling {sample_size} rows.")
        
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

        self.input_ids = []
        self.attention_masks = []
        self.labels = [] # For target summaries

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing data"):
            article = str(row['article'])
            highlight = str(row['highlights'])

            # Tokenize article (source)
            tokenized_article = self.tokenizer(
                article,
                max_length=self.max_source_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            self.input_ids.append(tokenized_article['input_ids'].squeeze(0))
            self.attention_masks.append(tokenized_article['attention_mask'].squeeze(0))

            # Tokenize highlight (target/labels)
            # For summarization, labels should also be padded/truncated
            tokenized_highlight = self.tokenizer(
                highlight,
                max_length=self.max_target_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            self.labels.append(tokenized_highlight['input_ids'].squeeze(0))

        self.input_ids = torch.stack(self.input_ids)
        self.attention_masks = torch.stack(self.attention_masks)
        self.labels = torch.stack(self.labels)

        print(f"Loaded {len(self.input_ids)} samples.")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx]
        }

def train_model(model, dataloader, optimizer, device):
    model.train()
    epoch_loss = 0
    pbar = tqdm(dataloader, desc="Training")

    for i, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRADIENT)

        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return epoch_loss / len(dataloader)

def evaluate_model(model, dataloader, device):
    model.eval()
    epoch_loss = 0
    pbar = tqdm(dataloader, desc="Validation")

    with torch.no_grad():
        for i, batch in enumerate(pbar):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            epoch_loss += loss.item()
            pbar.set_postfix(val_loss=loss.item())

    return epoch_loss / len(dataloader)


if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Supervised BART models handle padding token ID internally,
    # but we will get it for explicit use if needed (e.g., in generation)
    PAD_TOKEN_ID = tokenizer.pad_token_id
    if PAD_TOKEN_ID is None:
        PAD_TOKEN_ID = tokenizer.eos_token_id # Common fallback for BART if no explicit pad_token

    print(f"BART Tokenizer loaded. PAD_TOKEN_ID: {PAD_TOKEN_ID}")
    
    # Assuming normalized CSV files are available after running normalize.py
    train_csv_path = f"{PROCESSED_DATA_DIR}/train_normalized.csv"
    val_csv_path = f"{PROCESSED_DATA_DIR}/validation_normalized.csv"

    train_dataset = SummarizationDatasetBART(train_csv_path, tokenizer, MAX_SOURCE_LEN, MAX_TARGET_LEN, sample_size=20000)
    val_dataset = SummarizationDatasetBART(val_csv_path, tokenizer, MAX_SOURCE_LEN, MAX_TARGET_LEN, sample_size=20000)

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = BartSummarizer(model_name=MODEL_NAME, pad_token_id=PAD_TOKEN_ID).to(DEVICE)

    print(f"Number of parameters in model: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    optimizer = optim.AdamW(model.parameters(), lr=5e-5) # AdamW is common for Transformers

    best_val_loss = float('inf')

    print("\nStarting Supervised BART Training...")
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

        train_loss = train_model(model, train_dataloader, optimizer, DEVICE)
        val_loss = evaluate_model(model, val_dataloader, DEVICE)

        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Save the entire model state dictionary
            torch.save(model.state_dict(), f"{CHECKPOINTS_DIR}/bart_supervised_best_model.pt")
            print(f"  Saved best model with validation loss: {best_val_loss:.4f}")

    print("\nSupervised BART training complete!")


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Validation: 100%|██████████| 836/836 [03:28<00:00,  4.00it/s, val_loss=2.09]


  Train Loss: 2.1574
  Val Loss: 2.8091

Epoch 4/10


Validation: 100%|██████████| 836/836 [03:29<00:00,  4.00it/s, val_loss=2.22]


  Train Loss: 1.8356
  Val Loss: 2.9116

Epoch 5/10


Training:  78%|███████▊  | 978/1250 [13:14<03:40,  1.23it/s, loss=1.89]